In [1]:
import pandas as pd

df = pd.read_csv('bemo_prompts_dataset.csv')

In [2]:
df.sample(5)

,prompt,general,smart_home,todo,mail
229,"Start the vacuum cleaner, add 'finish the proj...",0,1,1,1
163,Remember to call mom tomorrow and send an upda...,0,0,1,1
12,How do I improve my productivity?,1,0,0,0
171,"How's the weather today, dim the bedroom light...",1,1,1,0
237,"Open the curtains in the living room, add a ta...",0,1,1,1


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   prompt      250 non-null    object
 1   general     250 non-null    int64 
 2   smart_home  250 non-null    int64 
 3   todo        250 non-null    int64 
 4   mail        250 non-null    int64 
dtypes: int64(4), object(1)
memory usage: 9.9+ KB


In [4]:
# Shuffle the data
df = df.sample(frac=1).reset_index(drop=True)

In [5]:
prompts = df['prompt'].values.tolist()
prompts[:5]

['Activate the home surveillance system and schedule a conference call for Monday.',
 'Lock the front door.',
 'Close the window blinds.',
 "What's the best movie tonight, open the curtains in the living room, and email the summary of the meeting.",
 'Can you tell me a fun fact, lock the front door, remind me to submit the report, and forward the latest report via email.']

In [6]:
labels = df.drop(columns=["prompt"]).values.tolist()
labels[:5]

[[0, 1, 1, 0], [0, 1, 0, 0], [0, 1, 0, 0], [1, 1, 0, 1], [1, 1, 1, 1]]

In [7]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

device = "cuda"
tokenizer = AutoTokenizer.from_pretrained("humarin/chatgpt_paraphraser_on_T5_base")
model = AutoModelForSeq2SeqLM.from_pretrained("humarin/chatgpt_paraphraser_on_T5_base").to(device)

In [8]:
def paraphrase(
    question,
    num_beams=5,
    num_beam_groups=5,
    num_return_sequences=5,
    repetition_penalty=1.5,
    diversity_penalty=0.5,
    no_repeat_ngram_size=2,
    temperature=0.7,
    max_length=128
):
    input_ids = tokenizer(
        f'paraphrase: {question}',
        return_tensors="pt", padding="longest",
        max_length=max_length,
        truncation=True,
    ).input_ids.to(device)
    
    outputs = model.generate(
        input_ids, temperature=temperature, repetition_penalty=repetition_penalty,
        num_return_sequences=num_return_sequences, no_repeat_ngram_size=no_repeat_ngram_size,
        num_beams=num_beams, num_beam_groups=num_beam_groups,
        max_length=max_length, diversity_penalty=diversity_penalty
    )

    result = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    return result

In [9]:
paraphrase("What is the capital of France?")

/home/momad/projects/BEMO/.venv/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


['What city serves as the capital of France?',
 'What is the name of the capital city in France?',
 'In which city is the capital of France located?',
 'Which city is designated as the capital of France?',
 'What is the name of the city that serves as the capital of France?']

In [10]:
paraphrased_prompts = []
paraphrased_labels = []

for i, prompt in enumerate(prompts):
    print(f"Processing prompt {i+1}/{len(prompts)}")
    results = paraphrase(prompt)
    paraphrased_prompts.extend(results)
    for _ in range(len(results)):
        paraphrased_labels.append(labels[i])

Processing prompt 1/250
Processing prompt 2/250
Processing prompt 3/250
Processing prompt 4/250
Processing prompt 5/250
Processing prompt 6/250
Processing prompt 7/250
Processing prompt 8/250
Processing prompt 9/250
Processing prompt 10/250
Processing prompt 11/250
Processing prompt 12/250
Processing prompt 13/250
Processing prompt 14/250
Processing prompt 15/250
Processing prompt 16/250
Processing prompt 17/250
Processing prompt 18/250
Processing prompt 19/250
Processing prompt 20/250
Processing prompt 21/250
Processing prompt 22/250
Processing prompt 23/250
Processing prompt 24/250
Processing prompt 25/250
Processing prompt 26/250
Processing prompt 27/250
Processing prompt 28/250
Processing prompt 29/250
Processing prompt 30/250
Processing prompt 31/250
Processing prompt 32/250
Processing prompt 33/250
Processing prompt 34/250
Processing prompt 35/250
Processing prompt 36/250
Processing prompt 37/250
Processing prompt 38/250
Processing prompt 39/250
Processing prompt 40/250
Processin

In [11]:
print(len(paraphrased_prompts), len(paraphrased_labels))

1250 1250


In [12]:
paraphrased_prompts[10:15]

['Close the window blinds.',
 'Close the window blinds.',
 'Open the window curtains.',
 'Close the window blinds.',
 'Remove the window curtains.']

In [13]:
paraphrased_labels[10:15]

[[0, 1, 0, 0], [0, 1, 0, 0], [0, 1, 0, 0], [0, 1, 0, 0], [0, 1, 0, 0]]

In [14]:
para_df = pd.DataFrame({
    "prompt": paraphrased_prompts,
    "general": [l[0] for l in paraphrased_labels],
    "smart_home": [l[1] for l in paraphrased_labels],
    "todo": [l[2] for l in paraphrased_labels],
    "mail": [l[3] for l in paraphrased_labels],
})

In [15]:
para_df.sample(5)

,prompt,general,smart_home,todo,mail
222,"What is the definition of quantum physics, wha...",1,1,1,0
1108,Send an email to invite .,0,0,0,1
379,"Adjust the dining room lighting, schedule a cl...",0,1,1,1
1057,How does meditation and relaxing music contrib...,1,1,0,0
948,"Tell me what's the latest update, set the ther...",1,1,0,1


In [16]:
import os
from dotenv import load_dotenv, find_dotenv, dotenv_values

load_dotenv()
os.environ["HUGGINGFACEHUB_API_TOKEN"] = dotenv_values(find_dotenv())[
    "HUGGINGFACE_API_KEY_TEST"
]
gemini_api_key = dotenv_values(find_dotenv())["GEMINI_API_KEY_TEST"]
os.environ["GOOGLE_API_KEY"] = gemini_api_key

In [17]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
